In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: rebuild from the raw CSV through cleaning, the split, and every embedding, then save those results.
# False: skip that work and load the saved embedding frames from data/processed/embeddings.
REBUILD = False

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split_and_weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned)
    df_engineered.cache().count()
    df_train, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_test = encoder(df_train, df_test)
    data_oi.save_split_data(df_train,df_test)


In [ ]:
from src.embeddings import EmbeddingModel

if REBUILD:
    embedding_model = EmbeddingModel(df_train,df_test)
    for name, train_function in EmbeddingModel.train_registry.items():
        print(f"Embedding Model `{name}` Training")
        train_function(embedding_model)
    train_embedding_dict = {}
    test_embedding_dict = {}
    for name, predict_function in EmbeddingModel.predict_registry.items():
        print(f"Embedding Model `{name}` Running")
        train_embedding_dict[name], test_embedding_dict[name] = predict_function(embedding_model)
        data_oi.save_embedding_data(name,train_embedding_dict[name],test_embedding_dict[name])

else:
    train_embedding_dict, test_embedding_dict = data_oi.load_embedding_data()


In [ ]:
from src.analytics import AnalyticsModel

analytics_name = "lr"

analytics_models = {}
for embedding_name in train_embedding_dict.keys():
    analytics_models[embedding_name] = AnalyticsModel(
        train_embedding_dict[embedding_name],
        test_embedding_dict[embedding_name]
    )

    print(f"Embedding Model `{embedding_name}` Analytics Model `{analytics_name}` Training")
    AnalyticsModel.train_registry[analytics_name](
        analytics_models[embedding_name],
        embedding_name
    )


In [ ]:
train_pred_dict = {}
test_pred_dict = {}

for embedding_name in train_embedding_dict.keys():
    print(f"Embedding Model `{embedding_name}` Analytics Model `{analytics_name}` Running")
    # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.

    model_name = f"{analytics_name}__{embedding_name}"
    train_pred_dict[model_name], test_pred_dict[model_name] = AnalyticsModel.predict_registry[analytics_name](
        analytics_models[embedding_name],
        embedding_name
    )
    data_oi.save_prediction_data(
        model_name,
        train_pred_dict[model_name],
        test_pred_dict[model_name]
    )


In [ ]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer = Scorer(test_pred_dict)
scorer.show_metrics(truncate=80)
